In [2]:
import torch
from esm.models.esmc import EsmcForMaskedLM, EsmcTokenizer

ESMC: Transformer Engine is not installed; falling back to pure-PyTorch LayerNorm+Linear / LayerNorm+MLP. Outputs will differ numerically - measured on the unnormalized residual stream (before the final LayerNorm), ~O(10) max-diff in fp32 and ~O(100) in bf16; after the final LayerNorm these shrink to a few ULP and perplexity stays within rounding noise. Install with `pip install transformer-engine[pytorch]` to enable fused fp32-reduction LayerNorm.
ESMC: neither xformers nor flash-attn is installed; falling back to PyTorch ``F.scaled_dot_product_attention``. The attention reduction order in bf16 differs from a fused kernel by ~1 bf16 ULP per attention block; compounded across the 80-block stack this reaches ~O(100) max-diff on the unnormalized residual stream. Install an xformers build matching your PyTorch and CUDA versions for a fused attention kernel.
ESMC: flash-attn rotary kernel not installed; falling back to pure-PyTorch RoPE. For faster GPU inference run `pip install flash-attn

In [3]:
import torch.nn.functional as F

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
# ESMC-300M Pretrained model
model = EsmcForMaskedLM.from_pretrained(
    "biohub/ESMC-300M",
    device=device,
).eval()

tokenizer = EsmcTokenizer()

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [6]:
def get_protein_embedding(amino_acid_seq):

    inputs = tokenizer([amino_acid_seq], return_tensors="pt", padding=True)
    inputs = {name: value.to(model.device) for name, value in inputs.items()}

    with torch.inference_mode():
        result = model(**inputs, output_hidden_states=True)

    token_embeddings = result.hidden_states[-1]    
    residue_embeddings = token_embeddings[0, 1:-1] # exlude start and end token

    return residue_embeddings


In [7]:
pcv_seq = "MPSKKNGRSGPQPHKRWVFTLNNPSEDERKKIREPPISLFDYFIVGEEGNEEGRTPHLQGFANFVKKQTFNKVKWYLGARCHIEKAKGTDQQNKEYCSKEGNLLIECGAPRSQGQRSDLSTAVSTLLESGSLVTVAEQHPVTFVRNFRGLAELLKVSGKMQKRDWKTNVHVIVGPPGCGKSKWAANFADPETTYWKPPRNKWWDGYHGEEVVVIDDFYGWLPWDDLLRLCDRYPLTVETKGGTVPFLARSILITSNQTPLEWYSSTAVPAVEALYRRITSLVFWKNATEQSTEEGGQFVTLSPPCPEFPYEINY"
wdv_seq = "MASSSAPRFRVYSKYLFLTYPECTLEPQYALDSLRTLLNKYEPLYIAAVRELHEDGSPHLHVLVQNKLRASITNPNALNLRMDTSPFSIFHPNIQAAKDCNQVRDYITKEVDSDVNTAEWGTFVAVSTPGRKDRDADMKQIIESSSSREEFLSMVCNRFPFEWSIRLKDFEYTARHLFPDPVATYTPEFPTESLICHETIESWKNEHLYSESPGRHKSIYICGPTRTGKTSWARSLGTHNYYNSLVDFTTYDVNAKYNIIDDIPFKFTPNWKCFVGAQRDFTVNPKYGKRKVIRGGIPCIILVNPDEDWLKDMTPEQSDYMYSNTVVHYMYEGETFINYSFASGEDVTASQ"
fbnyv_seq = "MARQVICWCFTLNNPLSPLSLHDSMKYLVYQTEQGEAGNIHFQGYIEMKKRTSLAGMKKLIPGAHFEKRRGTQGEARAYSMKEDTRLEGPWEYGEFVPTIEDKLREVMNDMKITGKRPIEYIEECCNTYDKSASTLREFRGELKKKKAISSWELQRKPWMGEVDALLQERDGRRIIWVYGPQGGEGKTSYAKHLVKTRDAFYSTGGKTADIAFAWDHQELVLFDFPRSFEEYVNYGVIEQLKNGIIQSGKYQSVIKYSDYVEVIVFANFTPRSGMFSEDRIVYVYA"

pcv_embed = get_protein_embedding(pcv_seq)
wdv_embed = get_protein_embedding(wdv_seq)
fbnyv_embed = get_protein_embedding(fbnyv_seq)

wdv_sDBM = wdv_embed[90:99, :]
pcv_sDBM = pcv_embed[78:87, :]
fbnyv_sDBM = fbnyv_embed[62:71, :]


In [8]:
def query_sDBM(protein_seq, known_sDBM):

    protein_embed = get_protein_embedding(protein_seq)

    length, _ = protein_embed.shape

    max_mean = -1
    next_max_mean = -1

    start_seq = 0
    end_seq = 0 

    for i in range(30, 150):
        match_block = protein_embed[i:i+9, :]

        sim = F.cosine_similarity(match_block, known_sDBM, dim = 1)

        mean = torch.mean(sim)

        # print(i + 1)
        # print(mean)

        if mean > max_mean:
            start_seq = i + 1
            end_seq = i + 9
            next_max_mean = max_mean
            max_mean = mean
        elif mean > next_max_mean:
            next_max_mean = mean
            next_start = i + 1
            next_end = i + 9

    if ((max_mean - next_max_mean) < 0.01):
        print("Unable to suggest sDBM... confidence measure below threshold")
        return

  #  diff = ((max_mean - next_max_mean) / max_mean) * 100
    perc = (max_mean - next_max_mean) * 100


    print("sDBM candidate found!!!")
    print(f"Max Mean Similarity: {max_mean}")
    print(f"Closest Mean Similarity to sDBM Prediction: {next_max_mean}")
    print(f"sDBM prediction from {start_seq} to {end_seq}")
    print(f"{next_start} to {next_end}")
    print(protein_seq[start_seq - 1: end_seq])
    print(perc)

In [ ]:
query_sDBM(pcv_seq, wdv_sDBM)
query_sDBM(fbnyv_seq, pcv_sDBM)
query_sDBM(fbnyv_seq, wdv_sDBM)